In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
# ─────────────────────────────────────────────────────
# Cell 1: Imports
# ─────────────────────────────────────────────────────
from faker import Faker
import random, json, uuid
import pandas as pd
from datetime import datetime

fake = Faker()
today = datetime.now()
year  = today.strftime("%Y")
month = today.strftime("%m")
day   = today.strftime("%d")

In [0]:
%run ../configs/common_configs

In [0]:
%run ../configs/log_configs

In [0]:
# ─────────────────────────────────────────────────────
# Cell 2: Widget Parameters (ADF-injected)
# ─────────────────────────────────────────────────────
dbutils.widgets.text("environment",    "dev")
dbutils.widgets.text("layer",          "raw")
dbutils.widgets.text("domain",         "ingestion_logs")
dbutils.widgets.text("source_name",    "")
dbutils.widgets.text("raw_path",       "")
dbutils.widgets.text("notebook_name",  "")
dbutils.widgets.text("pipeline_name",  "")
dbutils.widgets.text("file_format",    "csv")
dbutils.widgets.text("schema_config",  "")

environment   = dbutils.widgets.get("environment")
layer         = dbutils.widgets.get("layer")
domain        = dbutils.widgets.get("domain")
source_name   = dbutils.widgets.get("source_name")
raw_path      = dbutils.widgets.get("raw_path")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")
file_format   = dbutils.widgets.get("file_format")
schema_config = dbutils.widgets.get("schema_config")

print(f"Source        : {source_name}")
print(f"Schema Config : {schema_config}")
print(f"Environment   : {environment}")

In [0]:
# ─────────────────────────────────────────────────────
# Cell 3: Schema Loader
# Reads JSON from ADLS configs container
# ─────────────────────────────────────────────────────
def load_schema(schema_config_filename: str) -> dict:

    schema_path = (
        f"abfss://{environment}@{storage_account}"
        f".dfs.core.windows.net/{layer}/configs/schemas/"
        f"{schema_config_filename}"
    )

    schema_json = (
        spark.read
        .text(schema_path)
        .collect()
    )

    raw_text = "\n".join(
        [row.value for row in schema_json]
    )

    return json.loads(raw_text)

In [0]:
# ─────────────────────────────────────────────────────
# Cell 4: Generic Field Generator
# Resolves every field type defined in the JSON schema
# ─────────────────────────────────────────────────────
def resolve_field(field_spec: dict, record_context: dict) -> any:
    """
    Generates a single field value based on its spec type.
    record_context holds already-generated fields for derived types.

    Supported types:
        timestamp_rand   → "PFX" + YYYYMMDDHHMMSSffffff + rand(1000,9999)
        formatted_int    → "PFX" + zero-padded int
        random_float     → uniform float, optionally rounded
        random_choice    → uniform pick from choices list
        weighted_choice  → weighted pick from choices list
        fake_datetime    → Faker date_time_between, strftime formatted
        uuid             → uuid4 string
        lookup_map       → picks a key, then expands mapped sibling fields
        derived_flag     → 1/0 based on whether source_field == true_value
    """

    ftype = field_spec["type"]

    if ftype == "timestamp_rand":
        return (
            field_spec["prefix"]
            + datetime.now().strftime("%Y%m%d%H%M%S%f")
            + str(random.randint(1000, 9999))
        )

    elif ftype == "formatted_int":
        fmt  = field_spec["format"]
        val  = random.randint(field_spec["min"], field_spec["max"])
        return field_spec["prefix"] + format(val, fmt)

    elif ftype == "random_float":
        val = random.uniform(field_spec["min"], field_spec["max"])
        return round(val, field_spec.get("round", 2))

    elif ftype == "random_choice":
        return random.choice(field_spec["choices"])

    elif ftype == "weighted_choice":
        return random.choices(
            field_spec["choices"],
            weights=field_spec["weights"]
        )[0]

    elif ftype == "fake_datetime":
        return fake.date_time_between(
            start_date=field_spec["start_date"],
            end_date=field_spec["end_date"]
        ).strftime(field_spec["format"])

    elif ftype == "uuid":
        return str(uuid.uuid4())

    elif ftype == "derived_flag":
        src   = field_spec["source_field"]
        truth = field_spec["true_value"]
        return 1 if record_context.get(src) == truth else 0

    # lookup_map is handled at record level (see generate_records)
    # This branch should not normally be reached
    raise ValueError(
        f"Unsupported field type: '{ftype}' "
        f"for field '{field_spec.get('name')}'"
    )

In [0]:
# ─────────────────────────────────────────────────────
# Cell 5: Generic Record Generator
# Drives the full schema to produce N records
# ─────────────────────────────────────────────────────
def generate_records(schema: dict) -> list:

    records     = []
    record_count = schema.get("record_count", 1000)

    for _ in range(record_count):

        record = {}

        # ── ID field ──────────────────────────────
        id_spec          = schema["id_field"]
        record[id_spec["name"]] = resolve_field(id_spec, record)

        # ── Lookup/map field (expands to N sibling fields) ──
        if "lookup_field" in schema:

            lf          = schema["lookup_field"]
            chosen_key  = random.choice(lf["choices"])
            mapped_vals = lf["map"][chosen_key]

            record[lf["name"]] = chosen_key        # e.g. campaign_id
            record.update(mapped_vals)             # e.g. campaign_name, product_id

        # ── Regular fields ────────────────────────
        for field_spec in schema["fields"]:
            record[field_spec["name"]] = resolve_field(
                field_spec, record
            )

        records.append(record)

    return records

In [0]:
# ─────────────────────────────────────────────────────
# Cell 6: Execute — Load Schema, Generate, Write, Log
# Zero source-specific code from here on
# ─────────────────────────────────────────────────────
try:
    # ── 1. Load schema from ADLS ──────────────────
    schema = load_schema(schema_config)
    print(f"Schema loaded: {schema['source_name']} "
          f"| {schema['record_count']} records")

    # ── 2. Generate records ───────────────────────
    records  = generate_records(schema)
    df_pd    = pd.DataFrame(records)
    df_spark = spark.createDataFrame(df_pd)

    display(df_pd.head())

    # ── 3. Build partitioned ADLS path ───────────
    full_raw_path = (
        f"abfss://{environment}@{storage_account}"
        f".dfs.core.windows.net/"
        f"{layer}/{raw_path}/"
        f"year={year}/month={month}/day={day}/"
    )
    print(f"Writing to: {full_raw_path}")

    # ── 4. Write to ADLS ──────────────────────────
    (
        df_spark.write
        .mode("overwrite")
        .option("header", "true")
        .format(file_format)
        .save(full_raw_path)
    )

    # ── 5. Log SUCCESS ────────────────────────────
    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="SUCCESS",
        records_processed=df_spark.count(),
        message=(
            f"{source_name} generated and "
            f"stored in {layer} zone"
        ),
        environment=environment
    )

except Exception as e:

    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="FAILED",
        records_processed=0,
        message=f"{layer} Load Failed: {str(e)}",
        environment=environment
    )

    raise

In [0]:
%sql
SELECT * FROM bib_catalog.logs.generate_data_pipeline_logs
ORDER BY log_timestamp DESC
LIMIT 10